# NB09 — Graph Health: baseline topológico (Fase 3a)

Reflejo del **baseline topológico** bajo el plan ratificado por S5
(`docs/plans/graph_phase3_plan.vsm`, `@status ratified` 2026-10-11,
"si, ratifico.. podemos avanzar").

- **Capas** — el grafo se parte en 4 capas semánticas (nunca densidad
  global: mezcla tipos incompatibles):
  `estructura` (child_of/extracted_from/derived_from), `consenso`
  (supersedes/contradicts), `certificacion` (certified_by/answers),
  `correlacion` (correlates_with, no dirigida).
- **Métricas por capa** — densidad (dirigida/no-dirigida), P(k),
  clustering (transitividad local), Fiedler λ₂ (conectividad
  algebraica; honest-null si <2 nodos), betweenness top-N=10.
- **acyclic_check** — `child_of`/`derived_from`/`supersedes` deben ser
  DAG; un ciclo = inconsistencia del knowledge base (gate candidato 3c).
  `contradicts` es simétrico por diseño (2-ciclos esperados, no falla).
- **R10** — mismo grafo → mismo health byte-idéntico (orden fijo de
  capas/kinds/nodos, eigvalsh determinista, sin wall-clock en el
  cuerpo del snapshot).
- **NO-NEW-TRUTH** — `health()` es vista pura de `build()`; no muta el
  grafo, no escribe verdad nueva, rebuildable from scratch.

*Ejecutar desde la raíz del repo:* `python3 /tmp/opencode/run_notebook.py docs/papers/attractor_convergence/analysis/09_graph_health_baseline.ipynb`

In [1]:
import sys, json, hashlib
sys.path.insert(0, "packages/framework")
from epistemic_graph import build, health
from epistemic_graph.graph_health import write_snapshot, LAYERS

g = build()  # cache por fingerprint
m = g["meta"]
print("fingerprint :", m["fingerprint"][:16], "| cache_hit:", m["cache_hit"])
print("nodes:", m["nodes_total"], "| edges:", m["edges_total"])
print("edges_by_kind:", json.dumps(m["edges_by_kind"]))

fingerprint : 4d45dcc656f2f28f | cache_hit: False
nodes: 2075 | edges: 4748
edges_by_kind: {"answers": 713, "certified_by": 666, "child_of": 42, "contradicts": 1946, "correlates_with": 30, "derived_from": 19, "extracted_from": 1100, "supersedes": 232}


## ① Capas y denominadores por capa

Cada capa reporta **sus** `nodes`/`edges` (denominador honesto por capa,
nunca global). `correlacion` es no-dirigida: las aristas bidireccionales
`A→B` + `B→A` (ratified `<->`) colapsan a **una** arista no-dirigida.

In [3]:
h = health(g)
print(f"{'capa':<14} {'kinds':<40} {'n':>5} {'m':>5} {'dir?':>5}")
for name, kinds, directed in LAYERS:
    e = h["layers"][name]
    print(f"{name:<14} {','.join(kinds):<40} {e['nodes']:>5} {e['edges']:>5} {str(directed):>5}")
print()
print("sum edges across layers:", sum(h["layers"][n]["edges"] for n, _, _ in LAYERS),
      "| graph edges_total:", h["edges_total"])

capa           kinds                                        n     m  dir?
estructura     child_of,extracted_from,derived_from      1194  1161  True
consenso       supersedes,contradicts                     382  2178  True
certificacion  certified_by,answers                      1404  1379  True
correlacion    correlates_with                             20    15 False

sum edges across layers: 4733 | graph edges_total: 4748


## ② Métricas por capa — densidad, P(k), clustering, Fiedler λ₂

- **density** — dirigida: `|E| / (|V|·(|V|-1))`; no-dirigida: `|E| / (|V|·(|V|-1)/2)`.
- **clustering_avg** — transitividad local (Dirigidas: sobre proyección no-dirigida).
- **fiedler** — λ₂ del Laplaciano no-dirigido. Grafo **desconectado** → λ₂ ≈ 0
  (honest: no es conexo). `null` si <2 nodos.
- **betweenness_top** — top-10 por `(-centrality, id)` — orden estable R10.

In [5]:
print(f"{'capa':<14} {'rho':>10} {'C':>8} {'Fiedler':>10} {'bk_top':>6}")
for name, _, _ in LAYERS:
    e = h["layers"][name]
    print(f"{name:<14} {e['density']:>10.6g} {e['clustering_avg']:>8.4f} "
          f"{str(e['fiedler']):>10} {len(e['betweenness_top']):>6}")
print()
print("P(k) estructura:", h["layers"]["estructura"]["degree_p_k"][:8], "...")
print("P(k) consenso  :", h["layers"]["consenso"]["degree_p_k"][:8], "...")

capa                  rho        C    Fiedler bk_top
estructura     0.000815056   0.0000        0.0      9
consenso        0.0149648   0.5219        0.0     10
certificacion  0.000700067   0.0000        0.0      0
correlacion     0.0789474   0.0000        0.0      4

P(k) estructura: [[1, 1168], [2, 3], [3, 8], [4, 1], [5, 4], [6, 1], [7, 1], [9, 2]] ...
P(k) consenso  : [[1, 119], [2, 17], [3, 32], [4, 17], [5, 12], [6, 6], [7, 4], [8, 8]] ...


## ③ acyclic_check — gate candidato 3c

`child_of`, `derived_from`, `supersedes` **deben** ser acíclicos.
Un ciclo en cualquiera = inconsistencia del knowledge base.
`contradicts` NO entra aquí: es simétrico por diseño (2-ciclos esperados).

In [7]:
print(f"{'kind':<16} {'n':>5} {'m':>5} {'is_dag':>7} {'cyc_sccs':>9}")
for kind, e in h["acyclic_check"].items():
    print(f"{kind:<16} {e['nodes']:>5} {e['edges']:>5} {str(e['is_dag']):>7} {e['cyclic_sccs']:>9}")
assert all(e["is_dag"] for e in h["acyclic_check"].values()), "ACYCLIC VIOLATION"
print("\nacyclic_check: all DAG ✓")

kind                 n     m  is_dag  cyc_sccs
child_of            45    42    True         0
derived_from        38    19    True         0
supersedes         307   232    True         0

acyclic_check: all DAG ✓


## ④ R10 — mismo grafo → mismo health

Dos evaluaciones independientes del **mismo** grafo → snapshot byte-idéntico
(serialización `sort_keys=True`, sin wall-clock en el cuerpo).

In [9]:
h1 = health(g)
h2 = health(g)
b1 = json.dumps(h1, sort_keys=True).encode()
b2 = json.dumps(h2, sort_keys=True).encode()
same = hashlib.sha256(b1).hexdigest() == hashlib.sha256(b2).hexdigest()
print("R10 determinism:", same, "| bytes:", len(b1))
assert same, "R10 VIOLATION: same graph produced different health"

R10 determinism: True | bytes: 4082


## ⑤ Snapshot + reconstruction check (NO-NEW-TRUTH)

El snapshot es **vista derivada** del grafo: `write_snapshot` solo serializa;
`health()` no muta el grafo. Rebuild from scratch = mismo resultado.

In [11]:
# mutación check: health es pura
g_before = json.dumps(g, sort_keys=True)
_ = health(g)
g_after = json.dumps(g, sort_keys=True)
assert g_before == g_after, "NO-NEW-TRUTH: health mutated the graph"
print("health() pure: graph unchanged ✓")

# snapshot roundtrip
p = write_snapshot(h)
loaded = json.loads(p.read_text())
assert loaded == h
print("snapshot roundtrip:", p, "| bytes:", p.stat().st_size)

health() pure: graph unchanged ✓
snapshot roundtrip: /home/rmw3/vOSlab/state/epistemic/graph_health.json | bytes: 7819


## ⑥ Summary — baseline topológico de la fase 3a

Snapshot completo en `state/epistemic/graph_health.json` — el **mapa**
antes del motor de descubrimiento (3b).

In [13]:
summary = {
  "nb": "NB09 — graph health baseline (Fase 3a, plan ratified S5 2026-10-11)",
  "plan": "docs/plans/graph_phase3_plan.vsm @status ratified",
  "graph": {
    "fingerprint": m["fingerprint"][:16],
    "nodes_total": m["nodes_total"],
    "edges_total": m["edges_total"],
  },
  "layers": {
    name: {
      "nodes": h["layers"][name]["nodes"],
      "edges": h["layers"][name]["edges"],
      "density": h["layers"][name]["density"],
      "clustering": h["layers"][name]["clustering_avg"],
      "fiedler": h["layers"][name]["fiedler"],
    }
    for name, _, _ in LAYERS
  },
  "acyclic_check": {
    kind: {"is_dag": e["is_dag"], "cyclic_sccs": e["cyclic_sccs"]}
    for kind, e in h["acyclic_check"].items()
  },
  "checks": {
    "R10_determinism": "celda ④: same graph → same health hash",
    "NO-NEW-TRUTH": "celda ⑤: health() pure, graph unchanged",
    "acyclic_gate_candidate": "celda ③: child_of/derived_from/supersedes all DAG",
    "snapshot": "state/epistemic/graph_health.json (rebuildable)",
  },
}
assert all(e["is_dag"] for e in summary["acyclic_check"].values())
print(json.dumps(summary, indent=1, ensure_ascii=False))

{
 "nb": "NB09 — graph health baseline (Fase 3a, plan ratified S5 2026-10-11)",
 "plan": "docs/plans/graph_phase3_plan.vsm @status ratified",
 "graph": {
  "fingerprint": "4d45dcc656f2f28f",
  "nodes_total": 2075,
  "edges_total": 4748
 },
 "layers": {
  "estructura": {
   "nodes": 1194,
   "edges": 1161,
   "density": 0.000815056001,
   "clustering": 0.0,
   "fiedler": 0.0
  },
  "consenso": {
   "nodes": 382,
   "edges": 2178,
   "density": 0.014964752443,
   "clustering": 0.521868075918,
   "fiedler": 0.0
  },
  "certificacion": {
   "nodes": 1404,
   "edges": 1379,
   "density": 0.000700066808,
   "clustering": 0.0,
   "fiedler": 0.0
  },
  "correlacion": {
   "nodes": 20,
   "edges": 15,
   "density": 0.078947368421,
   "clustering": 0.0,
   "fiedler": 0.0
  }
 },
 "acyclic_check": {
  "child_of": {
   "is_dag": true,
   "cyclic_sccs": 0
  },
  "derived_from": {
   "is_dag": true,
   "cyclic_sccs": 0
  },
  "supersedes": {
   "is_dag": true,
   "cyclic_sccs": 0
  }
 },
 "checks": 